# Глава 13. Тексты: от TF-IDF до LLM — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Текст как данные: очистка и токены

Модели работают с числами, а отзывы гостей — это слова. Первый шаг любой работы с текстом — привести его к единому виду и разбить на слова-токены. Разбираемся с регистром, знаками препинания, частотами и стоп-словами — и с ловушкой: выкинуть «не» значит перевернуть смысл. В конце собираем первый мешок слов. Формы слов — в уроке «Леммы и словарь исключений».

Теория — на сайте курса: `ch13/01-text-prep.html`. Задачи «собери код из строк» (1) остались на сайте.

### Токенизатор · разминка

Напишите функцию `tokenize(text)`, которая возвращает список слов из русских букв в нижнем регистре. Сохраните `t = tokenize("Кофе — ОТЛИЧНЫЙ, ёжики!!! 10/10")`.

*Подсказки (2): `dsc.hint('tokenize')`*

In [ ]:
%%task ch13/01-text-prep/tokenize
import re

### Нормализация текста · разминка

Напишите `normalize(text)`: перевести в нижний регистр, заменить «ё» на «е», а все символы, кроме русских букв и пробела, заменить пробелами; затем схлопнуть повторяющиеся пробелы в один и убрать пробелы по краям. Примените к `raw` — результат в `clean`.

*Подсказки (3): `dsc.hint('ch13/01-text-prep/clean')`*

In [ ]:
%%task ch13/01-text-prep/clean
import re

def normalize(text):
    ...

clean = normalize(raw)
print(repr(clean))

### Словарь отзывов · разминка

Посчитайте частоты всех слов во всех отзывах (`Counter`). Сохраните `n_words` — число разных слов, `total` — общее число словоупотреблений и `top5` — список пяти самых частых слов (без частот).

*Подсказки (2): `dsc.hint('vocab')`*

In [ ]:
%%task ch13/01-text-prep/vocab
import re
from collections import Counter
import pandas as pd
r = pd.read_csv("reviews_text.csv")

### Слова плохих отзывов · база

Разделите отзывы на негативные (`rating <= 2`) и позитивные (`rating >= 4`). Посчитайте частоты слов в каждой группе. Сохраните `ne_share_neg` и `ne_share_pos` — долю отзывов с токеном «не» в каждой группе (3 знака) и `top_neg` — список из 10 слов, собранный так: идите по словам негативных отзывов от самых частых к редким и пропускайте те, что входят в 100 самых частых слов позитивных, пока не наберёте 10.

*Подсказки (2): `dsc.hint('neg-words')`*

In [ ]:
%%task ch13/01-text-prep/neg-words
import re
from collections import Counter
import pandas as pd
r = pd.read_csv("reviews_text.csv")

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

### Мешок слов руками · посложнее

Постройте мешок слов для трёх отзывов: словарь `vocab` — отсортированный список всех слов трёх отзывов, и матрицу `bow` (список списков или массив 3 × len(vocab)), где в строке i, столбце j — сколько раз слово `vocab[j]` встречается в отзыве i. Сохраните также `same_bag` — `True`, если у отзывов 0 и 2 одинаковые мешки.

*Подсказки (3): `dsc.hint('bag')`*

In [ ]:
%%task ch13/01-text-prep/bag
import re
docs = ["Кофе вкусный, но дорого.",
        "Кофе горький, кофе холодный.",
        "Дорого, но кофе вкусный!"]

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

### Редкие слова · база

Посчитайте частоты всех слов в отзывах (токенизатор `tokenize` — в заготовке). Сохраните `n_words` — размер словаря, `once` — отсортированный список слов, встретившихся ровно один раз, `share_once_vocab` — их долю среди **разных** слов словаря и `share_once_tokens` — их долю среди **всех** употреблений слов (обе доли — 4 знака).

*Подсказки (3): `dsc.hint('hapax')`*

In [ ]:
%%task ch13/01-text-prep/hapax
from collections import Counter

words = Counter()

### Пары слов в плохих отзывах · посложнее

Мешок слов теряет порядок, но пары соседних слов (**биграммы**) его частично сохраняют: «не понравился» — одна биграмма. Для каждого отзыва соберите множество его биграмм (`zip(tokens, tokens[1:])`). Посчитайте для каждой биграммы долю плохих (`rating <= 3`) и долю хороших отзывов, где она встречается. Сохраните `top_bad` — список из 5 биграмм (кортежей) с наибольшей разностью «доля в плохих − доля в хороших» (по убыванию), и `diff_top` — эту разность для первой из них (3 знака).

*Подсказки (3): `dsc.hint('bigrams')`*

In [ ]:
%%task ch13/01-text-prep/bigrams
from collections import Counter

bad = r["rating"] <= 3

### Правило «не» · со звёздочкой

Простейший классификатор тональности: «отзыв негативный, если в нём есть токен "не" **или** "но"». Посчитайте на всех отзывах (негативный — `rating <= 3`) долю правильных ответов `acc_rule` и F1 для класса «негативный» `f1_rule` (3 знака). Затем то же для правила только по «не» — `acc_ne`, `f1_ne`.

*Подсказки (2): `dsc.hint('ch13/01-text-prep/rule')` · задача по желанию*

In [ ]:
%%task ch13/01-text-prep/rule
import re
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

## Урок 2. Леммы и словарь исключений

«Сырники», «сырников» и «сырником» для модели — три разных слова. Сводим формы слова вместе: сначала грубо — обрезанием окончаний, потом по-настоящему — лемматизатором pymorphy3. Смотрим, где лемматизатор ошибается на словах меню, и чиним это словарём исключений.

Теория — на сайте курса: `ch13/01b-lemmas.html`.

### Стемминг обрезанием · база

Напишите функцию `stem(word, n=5)`: если слово длиннее `n` букв — оставить первые `n`, иначе вернуть как есть. Посчитайте число разных слов во всех отзывах до (`n_raw`) и после стемминга (`n_stem`). Сохраните `merged` — список слов, которые после стемминга совпали с `stem("курьер")`, в алфавитном порядке.

*Подсказки (2): `dsc.hint('stem')`*

In [ ]:
%%task ch13/01b-lemmas/stem
import re
import pandas as pd
r = pd.read_csv("reviews_text.csv")

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

### Словарь лемм · база

Размеры словаря до и после лемматизации (491 / 450 / 377) мы уже видели в теории. Теперь посмотрим, сколько работы лемматизатор сделал и насколько ему можно верить. Таблица `review_lemmas.csv` хранит лемму для каждого слова отзывов. Превратите её в словарь `to_lemma` («слово → лемма») и разбейте все отзывы на токены. Сохраните:

- `n_changed` — сколько **разных** слов отзывов лемматизатор изменил (лемма не совпадает с самим словом);
- `share_known` — долю **словоупотреблений** (каждого токена с повторами), которые есть в словаре pymorphy3 (`known == 1`), — 3 знака.

*Подсказки (3): `dsc.hint('lemmas')`*

In [ ]:
%%task ch13/01b-lemmas/lemmas
import re
import pandas as pd
r = pd.read_csv("reviews_text.csv")
lem = pd.read_csv("review_lemmas.csv")

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

### Исключения для слов меню · посложнее

Лемматизатор угадывает лемму незнакомого слова по окончанию и на словах меню ошибается. Найдите подозрительные слова: незнакомые словарю (`known == 0`) и частые (`count >= 30`). Сохраните их список `suspects` по убыванию `count`. Затем примените к **леммам** поправки `fixes` из заготовки (если лемма есть в `fixes`, заменить её) и посчитайте по всем токенам отзывов: `n_barista` — сколько раз встречается лемма `"бариста"` после поправки, `top_drink` — самая частая из лемм `"кофе"`, `"капучино"`, `"латте"`, `"раф"` после поправки.

*Подсказки (3): `dsc.hint('lemma-fix')`*

In [ ]:
%%task ch13/01b-lemmas/lemma-fix
import re
from collections import Counter
import pandas as pd
r = pd.read_csv("reviews_text.csv")
lem = pd.read_csv("review_lemmas.csv")
fixes = {"капучиный": "капучино", "латта": "латте", "барист": "бариста", "флэта": "флэт"}

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

## Урок 3. Мешок слов и TF-IDF

Превращаем отзывы в таблицу признаков: столбец на каждое слово, в клетке — сколько раз оно встретилось. Знакомимся с CountVectorizer и TfidfVectorizer, разреженными матрицами, биграммами и весом TF-IDF, который отличает характерные слова от просто частых.

Теория — на сайте курса: `ch13/02-tfidf.html`. Задачи «собери код из строк» (1) остались на сайте.

### Матрица счётчиков · разминка

Постройте `CountVectorizer` по всем отзывам. Сохраните `shape` (форма матрицы), `density` — долю ненулевых клеток (4 знака) и `n_kofe` — сколько раз слово «кофе» встречается во всех отзывах суммарно.

*Подсказки (2): `dsc.hint('ch13/02-tfidf/count')`*

In [ ]:
%%task ch13/02-tfidf/count
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
r = pd.read_csv("reviews_text.csv")

### Редкие и частые · разминка

Обучите `TfidfVectorizer` на всех отзывах и сохраните `idf` — Series весов idf с индексом-словами. В `common3` — три слова с наименьшим idf, в `idf_ne` — idf слова «не» (3 знака).

*Подсказки (2): `dsc.hint('idf')`*

In [ ]:
%%task ch13/02-tfidf/idf
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
r = pd.read_csv("reviews_text.csv")

### Формула idf · база

Проверьте формулу: для слова «доставка» посчитайте вручную `df` — в скольких отзывах оно встречается (используйте `CountVectorizer(binary=True)`, где клетка — 0 или 1), и idf = ln((1 + N) / (1 + df)) + 1. Сохраните `df_word`, `idf_manual` (4 знака) и `idf_sklearn` — из `TfidfVectorizer` (4 знака).

*Подсказки (2): `dsc.hint('idf-by-hand')`*

In [ ]:
%%task ch13/02-tfidf/idf-by-hand
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
r = pd.read_csv("reviews_text.csv")

### Размер словаря · база

Сравните размер словаря `CountVectorizer` на всех отзывах для трёх настроек: только слова; слова и биграммы; слова и биграммы с `min_df=5`. Сохраните словарь `sizes` с ключами `"1"`, `"1-2"`, `"1-2 min5"` и `has_ne_pridu` — есть ли среди признаков третьей настройки биграмма «не приду».

*Подсказки (2): `dsc.hint('ngrams')`*

In [ ]:
%%task ch13/02-tfidf/ngrams
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
r = pd.read_csv("reviews_text.csv")

### Сколько слов оставить · база

Параметр `min_df` выкидывает слова, которые встречаются меньше чем в `min_df` отзывах. Для `min_df` из `[1, 2, 5, 20]` обучите `CountVectorizer(min_df=...)` на всех отзывах и сохраните `vocab_by_min_df` — словарь `{min_df: размер словаря}`. Для `min_df=5` сохраните `kept_share` — какую долю всех употреблений слов (сумма матрицы) сохраняет урезанный словарь по сравнению с полным (4 знака), и `dropped_examples` — 5 первых по алфавиту слов, которые есть в полном словаре, но выпали при `min_df=5`.

*Подсказки (3): `dsc.hint('min-df')`*

In [ ]:
%%task ch13/02-tfidf/min-df
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
r = pd.read_csv("reviews_text.csv")

### Ключевые слова отзыва · посложнее

TF-IDF удобно использовать и без модели — как «ключевые слова» текста. Обучите `TfidfVectorizer()` на всех отзывах. Напишите `keywords(i, k=3)` — список из `k` слов отзыва номер `i` с наибольшим весом TF-IDF (по убыванию веса). Сохраните `kw` — словарь `{i: keywords(i)}` для `i` от 0 до 4 и `share_topic_word` — долю из первых 200 отзывов, у которых хотя бы одно из трёх ключевых слов совпадает с одним из слов-маркеров темы этого отзыва (словарь `MARKERS` — в заготовке).

*Подсказки (3): `dsc.hint('keywords')`*

In [ ]:
%%task ch13/02-tfidf/keywords
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
T = tf.fit_transform(r["text"])
names = tf.get_feature_names_out()

def keywords(i, k=3):
    ...

### Обучение векторайзера без утечки · посложнее

Разделите отзывы `train_test_split(r["text"], test_size=0.25, random_state=42)`. Обучите `TfidfVectorizer` только на обучающей части и преобразуйте обе. Сохраните `X_train`, `X_test`, `n_unseen` — сколько разных слов тестовой части нет в словаре векторайзера (токены — `re.findall(r"(?u)\b\w\w+\b", text.lower())`, так же, как в sklearn) и `empty_test` — число тестовых отзывов, у которых вектор целиком нулевой.

*Подсказки (3): `dsc.hint('train-only')`*

In [ ]:
%%task ch13/02-tfidf/train-only
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
r = pd.read_csv("reviews_text.csv")
text_train, text_test = train_test_split(r["text"], test_size=0.25, random_state=42)

### Символьные n-граммы · со звёздочкой

Сравните два векторайзера на опечатках. Обучите `TfidfVectorizer()` и `TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4))` на всех отзывах. Для пары «правильное слово — опечатка» `("выпечка", "ыпечка")` посчитайте косинусное сходство их TF-IDF-векторов (`sklearn.metrics.pairwise.cosine_similarity`) в обоих представлениях. Косинусное сходство меряет, насколько похожи два вектора: от 0 (ни одного общего признака) до 1 (векторы одинаковые по направлению); подробно оно разобрано в уроке «Эмбеддинги: векторы слов и отзывов». Что такое `char_wb`, сказано во врезке «Символьные n-граммы» раздела «Биграммы». Сохраните `sim_words`, `sim_chars` (3 знака) и `vocab_chars` — размер словаря символьного векторайзера.

*Подсказки (2): `dsc.hint('char-ngrams')` · задача по желанию*

In [ ]:
%%task ch13/02-tfidf/char-ngrams
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
r = pd.read_csv("reviews_text.csv")

## Урок 4. Классификация текстов

Соединяем TF-IDF и логистическую регрессию в один конвейер и учим модель отличать недовольных гостей от довольных, а потом — определять тему отзыва. Смотрим, на какие слова опирается модель, и разбираем её ошибки — среди них оказываются ошибки самих гостей. Как проверять саму разметку — в уроке «Качество разметки: голосование, каппа, золотой набор».

Теория — на сайте курса: `ch13/03-classify.html`. Задачи «собери код из строк» (1) остались на сайте.

### Конвейер тональности · разминка

Соберите конвейер `make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))`, обучите на обучающей части и сохраните `acc`, `f1` на тесте (3 знака) и `preds` — прогнозы для трёх новых отзывов из заготовки (список 0/1).

*Подсказки (2): `dsc.hint('ch13/03-classify/pipe')`*

In [ ]:
%%task ch13/03-classify/pipe
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)
X_train, X_test, y_train, y_test = train_test_split(r["text"], y, test_size=0.25, random_state=42, stratify=y)
new = ["Бариста грубил, кофе холодный", "Всё понравилось, спасибо", "Вкусно, но очень долго ждали"]

### С чем сравнивать · база

Прежде чем радоваться точности модели, сравните её с базовой. Обучите `DummyClassifier(strategy="most_frequent")` на тех же обучающих текстах и посчитайте на тесте `acc_dummy`, `f1_dummy` (`zero_division=0`) и те же метрики конвейера TF-IDF + логистическая регрессия — `acc_model`, `f1_model` (все — 3 знака). В `share_bad` сохраните долю недовольных во всех отзывах (3 знака).

*Подсказки (2): `dsc.hint('ch13/03-classify/baseline')`*

In [ ]:
%%task ch13/03-classify/baseline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)
X_train, X_test, y_train, y_test = train_test_split(r["text"], y, test_size=0.25, random_state=42, stratify=y)

### Слова-сигналы · база

Для конвейера из задачи «Конвейер тональности» (обучен в заготовке) сохраните `weights` — Series весов с индексом-словами, `top_bad` — 5 слов с наибольшим весом (по убыванию) и `top_good` — 5 с наименьшим (от самого отрицательного). В `w_ne` — вес слова «не» (2 знака).

*Подсказки (3): `dsc.hint('top-words')`*

In [ ]:
%%task ch13/03-classify/top-words
# уже есть: model — конвейер TF-IDF + логистическая регрессия, обученный на тональности
import pandas as pd

### Ошибки или неверные метки · разминка

Для той же модели найдите ошибки на тесте и разделите их на две группы: «подозрение на неверную метку» — модель уверена (вероятность класса, который она выбрала, больше 0.8), и остальные. Сохраните `n_errors`, `n_confident` и `confident_share` (3 знака).

*Подсказки (3): `dsc.hint('ch13/03-classify/errors')`*

In [ ]:
%%task ch13/03-classify/errors
# уже есть: model, X_test, y_test
import numpy as np

### Где модель сомневается · база

Для конвейера тональности (обучен в заготовке) посчитайте вероятности недовольства на тесте `p`. Разделите тестовые отзывы на «уверенные» (`p < 0.2` или `p > 0.8`) и «сомнительные» (остальные). Сохраните `share_unsure` — долю сомнительных (3 знака), `acc_sure` и `acc_unsure` — долю верных ответов (по порогу 0.5) в каждой группе (3 знака), и `unsure_examples` — тексты трёх сомнительных отзывов с вероятностью, ближайшей к 0.5.

*Подсказки (3): `dsc.hint('unsure')`*

In [ ]:
%%task ch13/03-classify/unsure
import numpy as np
import pandas as pd

p = model.predict_proba(X_test)[:, 1]

### Не пропустить жалобу · посложнее

Отдел заботы о гостях хочет видеть почти все жалобы: полнота класса «недоволен» на тесте — не ниже 0.95. Для конвейера тональности (обучен в заготовке) найдите наибольший порог `thr` из сетки `np.arange(0.05, 0.96, 0.01)`, при котором полнота ≥ 0.95 (округлите до 2 знаков). Сохраните `precision_at` — точность при этом пороге (3 знака), `n_flagged` — сколько тестовых отзывов отправится на разбор и `precision_05`, `recall_05` — точность и полноту при пороге 0.5 (3 знака).

*Подсказки (3): `dsc.hint('recall95')`*

In [ ]:
%%task ch13/03-classify/recall95
import numpy as np
from sklearn.metrics import precision_score, recall_score

p = model.predict_proba(X_test)[:, 1]

### Темы · посложнее

Обучите конвейер на темах (`r["topic"]`, разбиение со `stratify` по теме). Сохраните `acc_topic` (3 знака), `cm` — `pd.crosstab(y_test, pred)` и `worst_topic` — тему с наименьшей полнотой (доля правильно найденных отзывов этой темы).

*Подсказки (3): `dsc.hint('topics')`*

In [ ]:
%%task ch13/03-classify/topics
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
r = pd.read_csv("reviews_text.csv")
X_train, X_test, y_train, y_test = train_test_split(r["text"], r["topic"], test_size=0.25, random_state=42, stratify=r["topic"])

### Сравнение векторайзеров · со звёздочкой

Сравните на тональности кросс-валидацией (`cross_val_score`, `cv=5`, `scoring="f1"`, на обучающей части) четыре конвейера: TF-IDF слов + логрегрессия; TF-IDF слов и биграмм + логрегрессия; символьные n-граммы `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))` + логрегрессия; `CountVectorizer` + `MultinomialNB`. Сохраните словарь `cv_f1` с ключами `"words"`, `"bigrams"`, `"chars"`, `"nb"` (средние, 3 знака) и `best`.

*Подсказки (3): `dsc.hint('ch13/03-classify/compare')` · задача по желанию*

In [ ]:
%%task ch13/03-classify/compare
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)
X_train, X_test, y_train, y_test = train_test_split(r["text"], y, test_size=0.25, random_state=42, stratify=y)

## Урок 5. Качество разметки: голосование, каппа, золотой набор

Модель не бывает точнее своей разметки. Три сотрудника разметили одни и те же 450 отзывов — какую тему считать правильной, насколько они согласны и кому из них можно верить? Разбираем голосование большинством, каппу Коэна и золотой набор. Те же приёмы понадобятся, чтобы проверить разметку языковой моделью.

Теория — на сайте курса: `ch13/03b-labels.html`. Задачи «собери код из строк» (1) остались на сайте.

### Голосование большинством · база

Переведите `review_labels.csv` в широкую таблицу `wide` (строка — отзыв, столбцы — `аня`, `борис`, `вика`) и посчитайте итоговую тему голосованием: тема, за которую не меньше двух голосов, иначе `None`. Сохраните `vote` — Series с итогом (индекс — `review_id`), `n_no_majority` — число отзывов без большинства и `share_2of3` — долю отзывов, где согласны **ровно** двое (3 знака).

*Подсказки (3): `dsc.hint('ch13/03b-labels/vote')`*

In [ ]:
%%task ch13/03b-labels/vote
import pandas as pd
labels = pd.read_csv("review_labels.csv")

### Каппа Коэна руками · база

Напишите функцию `kappa(x, y)` — каппу Коэна для двух списков меток: $p_o$ — доля совпадений, $p_e$ — сумма по всем темам произведений долей темы у `x` и у `y`, итог $(p_o - p_e) / (1 - p_e)$. Посчитайте ей каппу для пар Аня–Борис (`kappa_ab`) и Аня–Вика (`kappa_av`), 3 знака. Сверьтесь с `cohen_kappa_score`.

*Подсказки (3): `dsc.hint('kappa')`*

In [ ]:
%%task ch13/03b-labels/kappa
import pandas as pd
from sklearn.metrics import cohen_kappa_score
wide = pd.read_csv("review_labels.csv").pivot(index="review_id", columns="annotator", values="label")

def kappa(x, y):
    x, y = list(x), list(y)
    ...

### Кому доверять: золотой набор · посложнее

Для 50 отзывов из `review_gold.csv` правильная тема известна. Посчитайте `gold_acc` — словарь «разметчик → доля верных на золотом наборе» (3 знака) и `worst` — имя худшего. Затем посчитайте `vote_gold` — долю верных у голосования трёх (отзыв без большинства — ошибка), 3 знака. Наконец, решите, что делать: сохраните `n_recheck` — сколько из 450 отзывов придётся перепроверять, если голос худшего разметчика не учитывать и брать тему, только когда два оставшихся согласны.

*Подсказки (3): `dsc.hint('ch13/03b-labels/golden')`*

In [ ]:
%%task ch13/03b-labels/golden
import pandas as pd
labels = pd.read_csv("review_labels.csv")
gold = pd.read_csv("review_gold.csv").set_index("review_id")["label"]
wide = labels.pivot(index="review_id", columns="annotator", values="label")

def majority(row):
    counts = row.value_counts()
    return counts.index[0] if counts.iloc[0] >= 2 else None

## Урок 6. Эмбеддинги: векторы слов и отзывов

Мешок слов не знает, что «латте» и «капучино» — почти одно и то же. Эмбеддинги превращают слово или целый отзыв в плотный вектор, где близкие по смыслу тексты оказываются рядом. Считаем косинусное сходство, строим простые эмбеддинги сами — LSA и идею word2vec, — ищем похожие отзывы и смотрим, где такой поиск ошибается.

Теория — на сайте курса: `ch13/04-embeddings.html`. Задачи «собери код из строк» (1) остались на сайте.

### Косинусное сходство · разминка

Напишите функцию `cosine(a, b)` и посчитайте `s_latte` — сходство «кофе» и «латте» и `s_late` — сходство «курьер» и «опоздал» (оба — числа, без округления). Словарь `vec` уже загружен.

*Подсказки (2): `dsc.hint('cos')`*

In [ ]:
%%task ch13/04-embeddings/cos
import numpy as np

def cosine(a, b):
    ...

s_latte = ...
s_late = ...

### Ближайшие слова · база

Для каждого слова из `vec`, кроме самого «кофе», посчитайте косинус с «кофе». Сохраните `sims` — Series (индекс — слова, значения — сходство), отсортированную по убыванию, и `top3` — список трёх ближайших слов.

*Подсказки (2): `dsc.hint('nearest')`*

In [ ]:
%%task ch13/04-embeddings/nearest
import numpy as np
import pandas as pd

### Похожие отзывы · база

Постройте LSA-эмбеддинги отзывов: `TfidfVectorizer()` на всех текстах, затем `TruncatedSVD(n_components=50, random_state=0)`. Нормируйте строки на длину 1. Сохраните `explained` — долю сохранённой дисперсии (2 знака), `Zn` — нормированную матрицу и `neighbours` — список `review_id` трёх отзывов, ближайших к отзыву `R-40002` (сам отзыв не включайте).

*Подсказки (2): `dsc.hint('lsa')`*

In [ ]:
%%task ch13/04-embeddings/lsa
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

r = pd.read_csv("reviews_text.csv")

### Поиск по кусочкам слов · база

Сравните два поисковика по отзывам: на словном TF-IDF (`TfidfVectorizer()`) и на символьном (`TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4))`), оба обучены на всех отзывах. Сходство запроса со всеми отзывами — `(T @ vec.transform([q]).T).toarray().ravel()` (строки TF-IDF уже нормированы). Напишите `search(vec, T, q, k=5)` — список текстов `k` самых похожих отзывов (порядок по убыванию сходства, `np.argsort(-sim, kind="stable")`). Для запроса `"курьер опоздал"` сохраните `top_words` и `top_chars`, а также `share_late_words` и `share_late_chars` — долю отзывов в каждом топ-5, где встречается подстрока `"опоздал"` (без учёта регистра).

*Подсказки (3): `dsc.hint('char-search')`*

In [ ]:
%%task ch13/04-embeddings/char-search
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

w_vec = TfidfVectorizer()
W = w_vec.fit_transform(r["text"])
c_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4))
C = c_vec.fit_transform(r["text"])

def search(vec, T, q, k=5):
    ...

### Сколько измерений нужно · посложнее

Обучите на тональности (`rating <= 3`, разбиение как в уроке «Классификация текстов») конвейеры `TfidfVectorizer() → TruncatedSVD(k, random_state=0) → LogisticRegression(max_iter=1000)` для `k` из `[10, 20, 50, 100]`. Сохраните `acc_by_k` — словарь `{k: доля правильных на тесте}` (3 знака), `acc_full` — то же для конвейера без SVD и `best_k` — `k` с наибольшей долей (при равенстве — меньшее).

*Подсказки (2): `dsc.hint('lsa-clf')`*

In [ ]:
%%task ch13/04-embeddings/lsa-clf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)
X_train, X_test, y_train, y_test = train_test_split(r["text"], y, test_size=0.25, random_state=42, stratify=y)

### Соседи слов для word2vec · база

Напишите функцию `skipgram_pairs(tokens, window=2)`, которая возвращает список пар `(центральное, сосед)` для всех слов в окне ±`window` (само слово себе не сосед). Посчитайте `n_pairs` — число пар по всем отзывам (токены — `tokenize` из заготовки, окно 2), `top_courier` — список трёх самых частых соседей слова «курьер» и `shared` — множество общих слов среди **трёх** самых частых соседей «дороговато» и «недорого».

*Подсказки (3): `dsc.hint('skipgram')`*

In [ ]:
%%task ch13/04-embeddings/skipgram
import re
from collections import Counter
import pandas as pd
r = pd.read_csv("reviews_text.csv")

def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

### Темы без обучения · со звёздочкой

Попробуйте определить тему отзыва вообще без разметки — сходством с описанием темы. Постройте LSA, как в задаче «Похожие отзывы» (TF-IDF и SVD обучены на всех текстах), переведите в то же пространство описания тем `desc` (`svd.transform(tfidf.transform(...))`), нормируйте и припишите каждому отзыву тему с наибольшим косинусом. Сохраните `zs_pred` (массив или Series тем), `zs_acc` — долю совпадений с колонкой `topic` (3 знака) и `base_acc` — долю самой частой темы (3 знака).

*Подсказки (3): `dsc.hint('zero-shot')` · задача по желанию*

In [ ]:
%%task ch13/04-embeddings/zero-shot
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

r = pd.read_csv("reviews_text.csv")

## Урок 7. Внутри трансформера: токены, маска, пулинг

Модели семейства BERT дают вектор каждому слову в его контексте. Саму сеть берут готовой, но путь «текст → вектор» аналитик проходит руками: разбираем токенизатор WordPiece, дополнение до общей длины, маску внимания и пулинг — и ловушку, из-за которой вектор отзыва зависит от соседей по пакету.

Теория — на сайте курса: `ch13/04b-transformers.html`. Задачи «собери код из строк» (1) остались на сайте.

### Токенизатор WordPiece · база

Напишите функцию `wordpiece(word)` по алгоритму из урока: с текущей позиции берётся самый длинный кусок из словаря `VOCAB` (не с начала слова — с префиксом `##`); если ни один кусок не подошёл — всё слово превращается в `["[UNK]"]`. Сохраните `pieces` — словарь «слово → список кусочков» для слов из `words` и `ids` — список id для фразы «курьер привезли холодный кофе» с `[CLS]` в начале и `[SEP]` в конце.

*Подсказки (3): `dsc.hint('wordpiece')`*

In [ ]:
%%task ch13/04b-transformers/wordpiece
VOCAB = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "кофе", "курьер", "##ом", "##а", "при", "##вез", "##ли",
         "холод", "##ный", "##ным", "кап", "##учино", "бар", "##иста", "не", "##вкус", "вкус", "##но", "раф"]
ID = {t: i for i, t in enumerate(VOCAB)}
words = ["курьером", "привезли", "холодным", "невкусно", "бариста", "латте", "кофейня"]

def wordpiece(word):
    pieces, start = [], 0
    ...

### Пулинг с маской · база

В заготовке — выход модели для трёх отзывов: `H` формы (3, 8, 5) — вектор на каждую из 8 позиций — и `attention_mask` формы (3, 8); настоящих токенов 8, 5 и 3, остальное — заполнители со «случайным мусором». Напишите функцию `mean_pool(H, mask)` — среднее только по настоящим токенам, форма результата (тексты, признаки). Сохраните `pooled = mean_pool(H, attention_mask)`, `naive = H.mean(axis=1)` и `cos_short` — косинус между `pooled[2]` и `naive[2]` (самый короткий отзыв, 3 знака). Затем проверьте устойчивость: в заготовке `H_long`, `mask_long` — тот же пакет, дополненный ещё 4 заполнителями. Сохраните `same` — `True`, если `mean_pool` даёт на нём тот же результат (`np.allclose`).

*Подсказки (3): `dsc.hint('masked-pool')`*

In [ ]:
%%task ch13/04b-transformers/masked-pool
# уже есть: H (3 × 8 × 5), attention_mask (3 × 8), H_long, mask_long
import numpy as np

def mean_pool(H, mask):
    ...

## Урок 8. Разметка текстов языковой моделью

Большая языковая модель размечает отзывы по описанию, без обучающих примеров. Разбираем, как написать инструкцию с закрытым списком меток, превратить ответ-строку в данные и проверить разметку на выборке с известными ответами, — и как запустить небольшую модель у себя на компьютере в LM Studio.

Теория — на сайте курса: `ch13/04c-llm.html`.

### Разбор ответов LLM · посложнее

В `raw` — восемь ответов языковой модели на промпт из урока. Разберите их в DataFrame `parsed` с колонками `sentiment`, `topic`, `ok` (по строке на ответ, в том же порядке). Правила: из ответа берётся текст от первой `{` до последней `}` и разбирается `json.loads`; ответ годен (`ok=True`), только если JSON разобрался, `sentiment` — из `positive`/`negative`, а `topic` — из шести тем. У негодных ответов `sentiment` и `topic` — `None`. Сохраните также `n_bad` — число негодных.

*Подсказки (2): `dsc.hint('parse-llm')`*

In [ ]:
%%task ch13/04c-llm/parse-llm
import json
import pandas as pd

def parse(answer):
    ...

### Проверка разметки LLM · посложнее

Прежде чем размечать все отзывы языковой моделью, её проверяют на размеченной выборке. В заготовке — `check` (200 отзывов с настоящей темой `topic`) и `llm_topic` — условные ответы модели на них (уже разобранные строки; `None` — ответ не разобрался). Сохраните `n_invalid` — число ответов `None` или не из шести тем, `acc_llm` — долю верных среди **всех** 200 (негодный ответ — ошибка), 3 знака, `recall_by_topic` — Series полноты по темам (3 знака) и `worst` — тему с наименьшей полнотой. В `better_than_model` — `True`, если `acc_llm` выше 0.79 (точность модели тем из урока «Классификация текстов»).

*Подсказки (3): `dsc.hint('llm-check')`*

In [ ]:
%%task ch13/04c-llm/llm-check
import pandas as pd

ans = pd.Series(llm_topic)
true = check["topic"]

## Урок 9. Проект: о чём жалуются гости

Финал главы. С сентября сеть собирает отзывы с карт — без оценок и без тем, просто тексты. Управляющая хочет каждую неделю видеть, где гости стали недовольнее и почему. Обучаем две модели на размеченных отзывах, размечаем ими новые, находим кофейню, где что-то сломалось, и причину — и поправляем долю негатива на ошибки самой модели.

Теория — на сайте курса: `ch13/05-project.html`.

### Шаг 1. Две модели · база

Оцените на отложенной четверти (`train_test_split(..., test_size=0.25, random_state=42, stratify=...)`) две модели вида `make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))`: тональность (цель `(rating <= 3).astype(int)`) и тему (цель `topic`). Сохраните `acc_sent` и `acc_topic` — доли правильных на тесте (3 знака). Затем обучите такие же модели **на всех** 2400 отзывах: `sent` и `topic_model`.

*Подсказки (2): `dsc.hint('ch13/05-project/models')`*

In [ ]:
%%task ch13/05-project/models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)

def new_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))

### Шаг 1б. Редкая тема: F1 вместо доли правильных · посложнее

Доля правильных у модели тем — около 82 %, но в отчёте управляющей будут **счётчики по темам**, и важна каждая тема, в том числе редкая. Доля правильных усредняет по отзывам: частая тема «кофе» тянет её вверх, а провал на редкой теме почти не виден. Поэтому смотрят F1 по каждому классу и **macro-F1** — простое среднее F1 по классам, где редкая тема весит столько же, сколько частая.

На том же разбиении, что в шаге 1 (тема, `stratify=r["topic"]`), обучите модель `new_model()` и сохраните:

- `f1_by_topic` — словарь «тема → F1 на тесте» (3 знака), `macro_f1` (3 знака);
- `rarest` — тему, которой меньше всего в тесте, и `worst` — тему с наименьшим F1;
- `count_ratio` — во сколько раз модель недосчитывает редчайшую тему: число отзывов, которые модель **назвала** этой темой, делённое на настоящее число в тесте (3 знака);
- `f1_rare_balanced` — F1 редчайшей темы у той же модели с `LogisticRegression(max_iter=1000, class_weight="balanced")` (3 знака).

*Подсказки (3): `dsc.hint('ch13/05-project/rare')`*

In [ ]:
%%task ch13/05-project/rare
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

r = pd.read_csv("reviews_text.csv")

def new_model(class_weight=None):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                         LogisticRegression(max_iter=1000, class_weight=class_weight))

X_train, X_test, t_train, t_test = train_test_split(
    r["text"], r["topic"], test_size=0.25, random_state=42, stratify=r["topic"])

### Шаг 2. Разметка новых отзывов · база

Модели `sent` и `topic_model` обучены в заготовке. Загрузите `reviews_new.csv` в `n` (дата — `datetime`) и добавьте колонки: `p_neg` — вероятность недовольства, `neg` — 1, если `p_neg >= 0.5`, иначе 0, `topic` — предсказанная тема, `period` — `"после"` для дат с 2026-10-05 включительно и `"до"` для остальных. Сохраните `n_after` — число отзывов в периоде «после».

*Подсказки (2): `dsc.hint('score')`*

In [ ]:
%%task ch13/05-project/score
import numpy as np
import pandas as pd

n = pd.read_csv("reviews_new.csv", parse_dates=["date"])

### Шаг 3. Уровень и рост · база

Посчитайте по точкам долю недовольных (`neg`) до и после 5 октября. Сохраните `by_shop` — DataFrame с индексом `shop_id` и колонками `до`, `после`, `рост` (`после − до`), все — 3 знака, отсортированный по `рост` по убыванию. В `worst_level` — точку с наибольшей долей «после», в `worst_growth` — с наибольшим ростом.

*Подсказки (2): `dsc.hint('ch13/05-project/shops')`*

In [ ]:
%%task ch13/05-project/shops
by_shop = ...

### Шаг 4. Почему и цитаты · посложнее

Таблица `n` размечена в заготовке. Для недовольных отзывов KZN-2 посчитайте структуру тем до и после: `kzn` — `pd.crosstab(topic, period, normalize="columns")`. То же для недовольных отзывов остальных пяти точек — `others`. Сохраните `cause` — тему, доля которой в жалобах KZN-2 выросла сильнее всего, и `quotes` — список из пяти текстов: недовольные отзывы KZN-2 «после» с темой `cause` и наибольшей `p_neg` (по убыванию).

*Подсказки (2): `dsc.hint('why')`*

In [ ]:
%%task ch13/05-project/why
bad = n[n["neg"] == 1]

### Шаг 5. С какой недели · посложнее

Таблица `n` размечена в заготовке. Добавьте `week` — понедельник недели (`n["date"].dt.to_period("W").dt.start_time`). Сохраните `weekly` — сводную таблицу долей недовольных: строки — недели, столбцы — точки. Найдите `jump_week` — неделю (Timestamp), в которую доля KZN-2 выросла сильнее всего по сравнению с предыдущей неделей, и `sept_max` — максимум недельной доли KZN-2 в неделях, начавшихся до 2026-10-01 (3 знака). Нарисуйте доли по неделям и выделите KZN-2 цветом — график не проверяется, но ступеньку на нём видно сразу.

*Подсказки (2): `dsc.hint('ch13/05-project/weekly')`*

In [ ]:
%%task ch13/05-project/weekly
import pandas as pd
import matplotlib.pyplot as plt

### Шаг 6. Поправка на ошибки модели · со звёздочкой

Доля «недовольных по модели» — не настоящая доля: модель пропускает часть недовольных и иногда ошибается в другую сторону. Если TPR — доля пойманных недовольных, а FPR — доля довольных, ошибочно названных недовольными, то наблюдаемая доля p = q · TPR + (1 − q) · FPR, откуда настоящая q = (p − FPR) / (TPR − FPR).

Оцените `tpr` и `fpr` на отложенной четверти размеченных данных (разбиение и модель — как в шаге 1, тональность). Затем исправьте доли KZN-2 до и после: `adj` — словарь `{"до": ..., "после": ...}` (3 знака). Таблица `n` размечена в заготовке.

*Подсказки (2): `dsc.hint('adjust')` · задача по желанию*

In [ ]:
%%task ch13/05-project/adjust
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

r = pd.read_csv("reviews_text.csv")
y = (r["rating"] <= 3).astype(int)
X_train, X_test, y_train, y_test = train_test_split(r["text"], y, test_size=0.25, random_state=42, stratify=y)

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()